In [24]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [25]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [26]:


queryProduct = """
SELECT 
    [ProductID],
    [Name],
    [ProductNumber],
    [WeightUnitMeasureCode]
      ,[FinishedGoodsFlag]
      ,[Color]
      ,[SafetyStockLevel]
      ,[ReorderPoint]
      ,[StandardCost]
      ,[ListPrice]
      ,[Size]
      ,[Weight]
      ,[DaysToManufacture]
      ,[ProductLine]
      ,[Class]
      ,[Style]
      ,[ProductSubcategoryID]
      ,[ProductModelID]
      ,[SellStartDate]
      ,[SellEndDate]
FROM Production.Product
"""

dimensionProducto = pd.read_sql_query(queryProduct, motorBaseDatos)



queryProductModel = """
SELECT 
  [ProductModelID],
  [Name]
FROM Production.ProductModel
"""

tablaProductModel = pd.read_sql_query(queryProductModel, motorBaseDatos)



queryProductPhoto = """
SELECT 
    [ProductPhotoID],
    [LargePhoto]
FROM Production.ProductPhoto
"""
tablaProductPhoto = pd.read_sql_query(queryProductPhoto, motorBaseDatos)
# tablaProductPhoto



queryProductProductPhoto = """
SELECT 
    [ProductID]
      ,[ProductPhotoID]
FROM Production.ProductProductPhoto
"""
tablaProductProductPhoto = pd.read_sql_query(queryProductProductPhoto, motorBaseDatos)
# tablaProductProductPhoto

# dimensionProducto
# tablaProductModel

dimensionProducto

,ProductID,Name,ProductNumber,WeightUnitMeasureCode,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate
0,1,Adjustable Race,AR-5381,None,False,None,1000,750,0.0000,0.00,None,NaN,0,None,None,None,NaN,NaN,2008-04-30,NaT
1,2,Bearing Ball,BA-8327,None,False,None,1000,750,0.0000,0.00,None,NaN,0,None,None,None,NaN,NaN,2008-04-30,NaT
2,3,BB Ball Bearing,BE-2349,None,False,None,800,600,0.0000,0.00,None,NaN,1,None,None,None,NaN,NaN,2008-04-30,NaT
3,4,Headset Ball Bearings,BE-2908,None,False,None,800,600,0.0000,0.00,None,NaN,0,None,None,None,NaN,NaN,2008-04-30,NaT
4,316,Blade,BL-2036,None,False,None,800,600,0.0000,0.00,None,NaN,1,None,None,None,NaN,NaN,2008-04-30,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,995,ML Bottom Bracket,BB-8107,G,True,None,500,375,44.9506,101.24,None,168.00,1,None,M,None,5.0,96.0,2013-05-30,NaT
500,996,HL Bottom Bracket,BB-9108,G,True,None,500,375,53.9416,121.49,None,170.00,1,None,H,None,5.0,97.0,2013-05-30,NaT
501,997,"Road-750 Black, 44",BK-R19B-44,LB,True,Black,100,75,343.6496,539.99,44,19.77,4,R,L,U,2.0,31.0,2013-05-30,NaT
502,998,"Road-750 Black, 48",BK-R19B-48,LB,True,Black,100,75,343.6496,539.99,48,20.13,4,R,L,U,2.0,31.0,2013-05-30,NaT


TRANSFORMACION

In [27]:
productPhoto = tablaProductProductPhoto.merge(tablaProductPhoto, on='ProductPhotoID')
productPhoto

,ProductID,ProductPhotoID,LargePhoto
0,1,1,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
1,2,1,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
2,3,1,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
3,4,1,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
4,316,1,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
...,...,...,...
499,995,1,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
500,996,1,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
501,997,102,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...
502,998,102,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...


In [28]:
dimensionProducto = dimensionProducto.merge(productPhoto, on='ProductID')


# Crear columnas nuevas con None y truncar, cuando tengan data asignada luego será necesario
for col, length in {
    "SpanishProductName": 50,
    "FrenchProductName": 50,
    "EnglishDescription": 400,
    "FrenchDescription": 400,
    "ChineseDescription": 400,
    "ArabicDescription": 400,
    "HebrewDescription": 400,
    "ThaiDescription": 400,
    "GermanDescription": 400,
    "JapaneseDescription": 400,
    "TurkishDescription": 400,
    "SizeRange": 50,
    "DealerPrice": 50,
    "Status": 7
}.items():
    dimensionProducto[col] = None
    dimensionProducto[col] = dimensionProducto[col].str[:length]




dimensionProducto.rename(columns={
    'ProductID' : 'ProductKey',
    'ProductNumber': 'ProductAlternateKey',
    'Name' : ' EnglishProductName',
    'SellStartDate' : 'StartDate',
    'SellEndDate' : 'EndDate',
    'ProductSubcategoryID' : 'ProductSubcategoryKey'
}, inplace=True)



# dimensionProducto
print(dimensionProducto.columns)


Index(['ProductKey', ' EnglishProductName', 'ProductAlternateKey',
       'WeightUnitMeasureCode', 'FinishedGoodsFlag', 'Color',
       'SafetyStockLevel', 'ReorderPoint', 'StandardCost', 'ListPrice', 'Size',
       'Weight', 'DaysToManufacture', 'ProductLine', 'Class', 'Style',
       'ProductSubcategoryKey', 'ProductModelID', 'StartDate', 'EndDate',
       'ProductPhotoID', 'LargePhoto', 'SpanishProductName',
       'FrenchProductName', 'EnglishDescription', 'FrenchDescription',
       'ChineseDescription', 'ArabicDescription', 'HebrewDescription',
       'ThaiDescription', 'GermanDescription', 'JapaneseDescription',
       'TurkishDescription', 'SizeRange', 'DealerPrice', 'Status'],
      dtype='object')


In [29]:
dimensionProducto = dimensionProducto.merge(tablaProductModel, on='ProductModelID', how='left')
dimensionProducto.rename(columns={
    'Name' : 'ModelName',
}, inplace=True)

print(dimensionProducto.columns)
# dimensionProducto


Index(['ProductKey', ' EnglishProductName', 'ProductAlternateKey',
       'WeightUnitMeasureCode', 'FinishedGoodsFlag', 'Color',
       'SafetyStockLevel', 'ReorderPoint', 'StandardCost', 'ListPrice', 'Size',
       'Weight', 'DaysToManufacture', 'ProductLine', 'Class', 'Style',
       'ProductSubcategoryKey', 'ProductModelID', 'StartDate', 'EndDate',
       'ProductPhotoID', 'LargePhoto', 'SpanishProductName',
       'FrenchProductName', 'EnglishDescription', 'FrenchDescription',
       'ChineseDescription', 'ArabicDescription', 'HebrewDescription',
       'ThaiDescription', 'GermanDescription', 'JapaneseDescription',
       'TurkishDescription', 'SizeRange', 'DealerPrice', 'Status',
       'ModelName'],
      dtype='object')


CARGAR A LA BODEGA

In [30]:
dimensionProducto.to_sql('dimensionProduct',motorBodegaDatos, if_exists='replace',index=False)

56